# Binary field inverse

Adapted from the complete eleven-cell Qualtran v0.7.0 notebook at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2024 Google LLC, Apache-2.0. Native signatures, names and graph tables use FlagQuantum owners.

In [ ]:
from IPython.display import SVG, display
from sympy import Symbol
from examples.algorithms.field_inverse_catalog import field_inverse_example, field_inverse_example_graph, field_inverse_example_gate_model
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

## Out-of-place binary field inversion

The input word x represents a field element in GF(2**m). The raw operation computes x**(2**m-2) through the Itoh–Tsujii chain, retaining x, a result word and intermediate junk words. Zero maps to zero. The recurrence B(i+j) = B(i) B(j)**(2**i) combines Frobenius powers and Karatsuba products. Squaring is a binary linear transformation.

`binary_field_encoding` owns numerical field construction. Compiler owns the big-endian register decomposition and caller-provided workspace. The x port is THRU. The result and junk ports are RIGHT. For a concrete degree m, the actual junk word count is bit_length(m-1)-2+max(popcount(m-1)-1,1). Degree one omits the junk port. The symbolic source uses 2*bit_length(m-1)-2. The original prose's m-2 shape differs from its actual implementation.

The raw operation returns live junk. Only recursive scratch is zeroed. `gf2_inverse_ir` uses compute-copy-uncompute to XOR the inverse into an arbitrary target and restore all workspace. The original SynthesizeLRCircuit adjoint defect remains in the frozen reference. Native execution uses exact linear adjoints. Repaired-reference gate parity and unchanged-source declaration parity are recorded separately in FIELD_INVERSE_FILE_AUDIT.md.

The source cites Amento et al., Efficient quantum circuits for binary elliptic curve arithmetic, Section 2.3 (2012), Itoh and Tsujii, Structure of parallel multipliers for a class of fields GF(2**m) (1989), and Concrete quantum cryptanalysis of binary elliptic curves, Algorithm 2.

In [ ]:
from flagquantum.simulation.finite_field_addition import binary_field_encoding
from flagquantum.simulation.finite_field_inverse import field_inverse_classical_values
from flagquantum.compiler.registers.finite_field.field_inverse import gf2_inverse_decomposition, gf2_inverse_ir, gf2_inverse_workspace_size
from examples.algorithms.binary_field_arithmetic import FieldOperation
import flagquantum as fq
import torch

### Example Instances

In [ ]:
concrete = field_inverse_example(4)
concrete

In [ ]:
symbolic = field_inverse_example(Symbol("m"))
symbolic

#### Graphical Signature

The original notebook displays the concrete degree-four signature. The unrestricted symbolic instance above retains its original assumptions.

In [ ]:
display(SVG(field_signature_svg([(concrete[0], concrete[1], {})])))

### Call Graph

All one-level declared children and leaf multiplicities are retained. Static output width, primitive gate counts and allocated execution workspace have separate accounts.

In [ ]:
graph = field_inverse_example_graph(concrete)
display(SVG(render_call_graph_svg(export_call_graph_dot(graph.root, graph.direct_calls))))
print("One-level declared leaves", graph.leaf_counts)
print("Degree-four source gate model", field_inverse_example_gate_model(4))

## Execute through append

This coherent degree-three example checks all complex input amplitudes and every raw result/junk output against the numerical callback, including zero. Unlisted physical wires remain zero. Reversing the primitive stream restores the complete initial state. The clean XOR example then exercises every input with a nonzero target and checks all workspace bits. These paths have one coherent branch and no measurements. Measurement-assisted erasure retains its separately accepted fixed-wire chain evidence.

The source model counts raw compute primitives. The clean wrapper performs two raw computations and one width-sized copy. Physical binding permutations remain explicit when needed by a consumer.

In [ ]:
from collections import Counter
encoding = binary_field_encoding(3)
width = 3
modulus = int(encoding.gf_type.irreducible_poly)
needed = gf2_inverse_workspace_size(width, modulus=modulus)
count = width + needed
plan = gf2_inverse_decomposition(tuple(range(width)), tuple(range(width, count)), modulus=modulus)
state = torch.zeros(1 << count, dtype=torch.complex128)
expected = torch.zeros_like(state)
for value in range(1 << width):
    state[value << needed] = complex(value + 1, value - 2)
    x, result, junk = field_inverse_classical_values(encoding, encoding.gf_type(value))
    bits = [0] * count
    for word, wires in zip([int(x), int(result), *(int(v) for v in junk)], plan.outputs, strict=True):
        for bit, wire in enumerate(reversed(wires)):
            bits[wire] = (word >> bit) & 1
    expected[int("".join(map(str, bits)), 2)] = state[value << needed]
expected /= state.norm()
state /= state.norm()
circuit = fq.Circuit(count, inputs=state, dtype=torch.complex128).append(FieldOperation(count, plan.instructions))
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=1e-12)
restored = circuit.append(FieldOperation(count, tuple(reversed(plan.instructions))))
torch.testing.assert_close(restored.state().flatten(), state, rtol=0, atol=1e-12)
print("Actual raw primitive counts", dict(Counter(g.name for g in plan.instructions)))
print("Independent raw source model", field_inverse_example_gate_model(width))
print("Raw logical outputs", plan.outputs, "zero scratch", plan.scratch)
count = 2 * width + needed
clean = gf2_inverse_ir(tuple(range(width)), tuple(range(width, 2 * width)), tuple(range(2 * width, count)), modulus=modulus)
state = torch.zeros(1 << count, dtype=torch.complex128)
expected = torch.zeros_like(state)
for value in range(1 << width):
    inverse = int(field_inverse_classical_values(encoding, encoding.gf_type(value))[1])
    for target in range(1 << width):
        initial = ((value << width) | target) << needed
        final = ((value << width) | (target ^ inverse)) << needed
        state[initial] = complex(value + 1, target - 3)
        expected[final] = state[initial]
expected /= state.norm()
state /= state.norm()
clean_circuit = fq.Circuit(count, inputs=state, dtype=torch.complex128).append(FieldOperation(count, clean))
torch.testing.assert_close(clean_circuit.state().flatten(), expected, rtol=0, atol=1e-12)
print("Actual clean primitive counts", dict(Counter(g.name for g in clean)))
print("Clean workspace returned to zero", needed, "measurements", 0)